In [5]:
import numpy as np
import pandas as pd
import pylahman as lahman

In [3]:
df = pd.read_csv('../elo_output.csv')

In [11]:
df[np.isclose(df['m1_0_pitcher_rating_pre'], 1700)].tail()

,Unnamed: 0,gid,batter,pitcher,date,pa_result,m1_0_batter_rating_pre,m1_0_batter_rating_post,m1_0_batter_k,m1_0_pitcher_k,m1_0_pitcher_rating_pre,m1_0_pitcher_rating_post,m1_1_batter_rating_pre,m1_1_batter_rating_post,m1_1_batter_k,m1_1_pitcher_k,m1_1_pitcher_rating_pre,m1_1_pitcher_rating_post
13915318,13915318,TEX201806080,deshd002,verlj001,2018-06-08,other_out,1508.336,1506.802,20.0,20.0,1699.993,1701.527,1557.575,1556.612,12.387,4.000,1746.309,1746.620
14092070,14092070,OAK201906010,davik003,verlj001,2019-06-01,other_out,1582.889,1580.677,20.0,20.0,1700.008,1702.219,1615.628,1614.013,13.878,4.000,1719.233,1719.699
14552293,14552293,NYN202207100,escoe001,alcas001,2022-07-10,other_out,1501.045,1499.637,20.0,20.0,1700.008,1701.416,1570.420,1570.009,4.000,4.000,1678.763,1679.174
14618731,14618731,CLE202209171,arral001,clase001,2022-09-17,other_out,1584.327,1582.329,20.0,20.0,1699.999,1701.998,1648.996,1648.546,4.000,18.444,1732.708,1734.786
14966761,14966761,CLE202408262,dejop001,clase001,2024-08-26,k,1587.038,1584.920,20.0,20.0,1700.009,1702.127,1603.185,1602.594,6.050,18.817,1736.929,1738.769


In [ ]:
# df.groupby('batter')['batter_rating_pre'].max().sort_values()
df_grouped = df.groupby('batter')['batter_rating_pre'].agg(
    min_rating='min',
    max_rating='max',
    apperances='count'
).reset_index()
df_grouped.sort_values('max_rating', ascending=False)

,batter,min_rating,max_rating,apperances
1103,bondb001,1499.151181,1970.008686,12839
7174,maysw101,1479.380978,1910.566073,12797
5721,judga001,1483.297626,1889.427566,5122
12203,willt103,1500.000000,1885.136931,6589
6897,mantm101,1489.736068,1874.058060,10022
...,...,...,...,...
6093,kriek101,1497.961858,1500.000000,2
6092,kreuf101,1416.315146,1500.000000,76
6087,kremj101,1491.913264,1500.000000,8
6084,krehj001,1500.000000,1500.000000,1


In [3]:
df_people = lahman.People()
df_people['name'] = df_people.apply(lambda row: f'{row['nameFirst']} {row['nameLast']}', axis=1)
df_people = df_people[['name', 'retroID']]
df_people

,name,retroID
0,David Aardsma,aardd001
1,Hank Aaron,aaroh101
2,Tommie Aaron,aarot101
3,Don Aase,aased001
4,Andy Abad,abada001
...,...,...
24265,Frank Zupo,zupof101
24266,Paul Zuvella,zuvep001
24267,George Zuverink,zuveg101
24268,Dutch Zwilling,zwild101


In [ ]:
df = pd.merge(df_grouped, df_people, left_on='batter', right_on='retroID')
df[['name', 'min_rating', 'max_rating', 'apperances']].sort_values('apperances', ascending=False)

,name,min_rating,max_rating,apperances
9746,Pete Rose,1468.870604,1777.630550,16354
12359,Carl Yastrzemski,1471.570242,1867.152309,14261
1,Hank Aaron,1485.318772,1821.739518,14159
4844,Rickey Henderson,1490.394617,1743.754933,13679
9039,Albert Pujols,1487.728291,1822.363049,13335
...,...,...,...,...
5306,Bill Hurst,1500.000000,1500.000000,1
11810,Dick Wantz,1500.000000,1500.000000,1
10137,Ken Schrom,1500.000000,1500.000000,1
3549,Tom Flanigan,1500.000000,1500.000000,1


In [4]:
df = pd.read_csv('../matchups.csv')
df = df[['batter', 'pitcher', 'pa_result', 'batter_rating_pre', 'batter_rating_post', 'pitcher_rating_pre', 'pitcher_rating_post']]
df

,batter,pitcher,pa_result,batter_rating_pre,batter_rating_post,pitcher_rating_pre,pitcher_rating_post
0,gracj101,hught102,other_out,1500.000000,1497.139433,1500.000000,1502.860567
1,lewib103,hught102,other_out,1500.000000,1497.165413,1502.860567,1505.695154
2,spens101,hught102,other_out,1500.000000,1497.191155,1505.695154,1508.504000
3,pelle101,wynne101,other_out,1500.000000,1497.139433,1500.000000,1502.860567
4,peskj101,wynne101,other_out,1500.000000,1497.165413,1502.860567,1505.695154
...,...,...,...,...,...,...,...
12254538,montc004,parkm001,other_out,1618.698546,1614.985054,1524.415431,1528.128924
12254539,vargm001,parkm001,other_out,1586.297049,1582.904500,1528.128924,1531.521473
12254540,housb001,cannj001,other_out,1488.349529,1485.446543,1483.198218,1486.101204
12254541,hassr002,cannj001,k,1495.198674,1491.872739,1486.101204,1489.427139


In [7]:
df_scores = df.copy(deep=True)
df_scores['pre_rtg_diff'] = df_scores['batter_rating_pre'] - df_scores['pitcher_rating_pre']
df_scores['post_rtg_diff'] = df_scores['batter_rating_post'] - df_scores['pitcher_rating_post']
df_scores = pd.merge(df_scores, df_people, left_on='batter', right_on='retroID').rename(columns={'name': 'batter_name'})
df_scores = df_scores.drop(columns=['retroID', 'batter'])
df_scores = pd.merge(df_scores, df_people, left_on='pitcher', right_on='retroID').rename(columns={'name': 'pitcher_name'})
df_scores = df_scores.drop(columns=['retroID', 'pitcher'])
df_scores
# df = pd.merge(df, df_people, left_on='pitcher', right_on='retroID').rename(columns={'name': 'pitcher_name'})

,pa_result,batter_rating_pre,batter_rating_post,pitcher_rating_pre,pitcher_rating_post,pre_rtg_diff,post_rtg_diff,batter_name,pitcher_name
0,other_out,1500.000000,1497.139433,1500.000000,1502.860567,0.000000,-5.721135,Joe Grace,Tex Hughson
1,other_out,1500.000000,1497.165413,1502.860567,1505.695154,-2.860567,-8.529741,Buddy Lewis,Tex Hughson
2,other_out,1500.000000,1497.191155,1505.695154,1508.504000,-5.695154,-11.312845,Stan Spence,Tex Hughson
3,other_out,1500.000000,1497.139433,1500.000000,1502.860567,0.000000,-5.721135,Eddie Pellagrini,Early Wynn
4,other_out,1500.000000,1497.165413,1502.860567,1505.695154,-2.860567,-8.529741,Johnny Pesky,Early Wynn
...,...,...,...,...,...,...,...,...,...
12297174,other_out,1618.698546,1614.985054,1524.415431,1528.128924,94.283115,86.856130,Colson Montgomery,Mitchell Parker
12297175,other_out,1586.297049,1582.904500,1528.128924,1531.521473,58.168125,51.383027,Miguel Vargas,Mitchell Parker
12297176,other_out,1488.349529,1485.446543,1483.198218,1486.101204,5.151312,-0.654661,Brady House,Jonathan Cannon
12297177,k,1495.198674,1491.872739,1486.101204,1489.427139,9.097469,2.445600,Robert Hassell,Jonathan Cannon


In [9]:

df_scores.sort_values('pre_rtg_diff', ascending=False)[['batter_name', 'pitcher_name', 'pre_rtg_diff', 'post_rtg_diff', 'pa_result']]

,batter_name,pitcher_name,pre_rtg_diff,post_rtg_diff,pa_result
7831386,Barry Bonds,Terry Mulholland,478.065343,479.323014,walk
8234527,Barry Bonds,Jeff Weaver,469.850250,471.003170,walk
7844089,Barry Bonds,Bobby Jones,465.879397,467.190688,walk
6249139,Barry Bonds,Mark Knudson,460.888067,448.693769,k
8234544,Barry Bonds,Jeff Weaver,460.679568,461.875430,walk
...,...,...,...,...,...
5561658,Kent Tekulve,Jay Howell,-455.435102,-455.439824,other_out
8139273,Steve Reed,Eric Gagne,-456.834584,-457.014595,other_out
8864962,David Weathers,Rafael Betancourt,-463.277989,-463.322180,other_out
8166478,Scott Sullivan,Billy Wagner,-472.316693,-472.424344,other_out
